<h1 style="color:#004080; font-size:2em; margin-bottom:0.3em;"> Preparing and Executing a NOAA NGEN Simulation</h1>
<h4 style="color:gray; font-weight:normal;">
The purpose of this notebook is to demonstrate how CIROH-developed tools can be used to construct and execute a model simulation within the NOAA Next Generation (NGEN) modeling framework.
</h4>



<div style="background-color: #f7f7f7; padding: 10px; border-radius: 5px;">
    <p><strong>Authors:</strong> 
    <ul>
        <li>Anthony Castronova, <a href="mailto:acastronova@cuahsi.org">acastronova@cuahsi.org</a></li>
        <li>Irene Garousi-Nejad, <a href="mailto:igarousi@cuahsi.org">igarousi@cuahsi.org</a></li>
    </ul>
    </p>
    <p><strong>Last Modified:</strong> 6/9/25</p>
    <p><strong>Description:</strong> This Jupyter notebook is designed as a tutorial to guide learners through the process of designing and executing a hydrologic modeling simulation within the NOAA Next Generation (NextGen) framework, using a real-world case study: USGS 02464000 – North River near Samantha, Alabama. In this tutorial, we will use two key components of the NextGen framework: 
    <ul>
        <li><strong>NextGen-in-a-Box (NGIAB) data preprocessing</strong> tools, which assist in preparing the input data required to run the model simulation.</li>
        <li>The precompiled <strong>NGEN executable</strong>, which is already installed in this environment and will be used to execute the simulation.</li>
    </ul>
    </p>
    <p><strong>Software Requirements:</strong> This notebook has been tested using Python v3.11.8 using the following library versions:
    <blockquote>
        <ul>
            <li>fiona==1.9.6</li>
            <li>folium==0.16.0</li>
            <li>geopandas==1.0.1</li>
            <li>matplotlib==3.8.3</li>
            <li>xarray==2024.2.0</li>
            <li>pandas==2.2.1</li>
            <li>dataretrieval==1.0.12</li>
            <li>numpy==1.26.4</li>
        </ul>
    </blockquote>
    </p>
    <p><strong>Supporting Files and Dependencies:</strong> 
        <ul>
            <li>CIROH 2i2c JupyterHub</li>
            <li>NGIAB Data Preprocessor</li>
        </ul>
    </p>
    <p><strong>References:</strong></p>
    <ul>
        <li>https://github.com/NOAA-OWP/NextGen-Info</li>
        <li>https://github.com/NOAA-OWP/ngen/blob/master/doc/T-shirt_model_description.pdf</li>
    </ul>
</div>

<i class="fas fa-code" style="color: black; margin-right: 8px;"></i>
<strong>Import Required Python Libraries</strong>

In [ ]:
import fiona
import folium
import geopandas as gpd
import matplotlib.pyplot as plt
import xarray
import pandas
import dataretrieval.nwis as nwis
import numpy as np

### Introduction to the NGEN Tools Used in this Notebook 

#### NGIAB Data Preprocessor
The NextGen-in-a-Box (NGIAB) Data Preprocessor tools are designed to assist in the prepartion of data needed to run NextGen model simulations. The tool uses geometry and model attributes from the v2.2 hydrofabric, along with raw forcing data that can be sourced from either the [National Water Model Retrospective v3](https://registry.opendata.aws/nwm-archive/), which u is provided in a projected Lambert Conformal Conic coordinate system with a 1 km spatial resolution, or the [AORC](https://registry.opendata.aws/noaa-nws-aorc/) gridded dataset, , which uses a geographic (longitude-latitude) coordinate system based on the World Geodetic System 1984 (WGS 84).

The `NGIAB Data Preprocessor` tools can be executed from the commandline using the sytax:

```
python -m <module name> <args>
```

for example

```
python -m ngiab_data_cli --help
```

#### NGEN Executable
The NGEN is a foundational, modular, and model-agnostic hydrologic modeling framework, meaning it supports the integration of various hydrologic models through standardized interfaces.

The `NGEN executable` can be run from the commandline using the sytax:

```
ngen <catchment_data_path> <catchment subset ids> all <nexus_data_path> <nexus subset ids> all <realization_config_path>
```

for example

```
ngen
```

<div style="border-left: 4px solid #007acc; background-color: #eef6fb; padding: 10px; border-radius: 4px;">
    <p> <strong>Summary:</strong>
        This notebook performs the following tasks using both tools described above: the NGIAB data preprocessor is used for tasks 1–3, and the NGEN executable is used to run the simulation in task 4.
    </p>
    <ol>
        <li>Subset (delineate) the NextGen HydroFabric based on a user-provided point of interest (e.g., catchment, gage, flowpath).</li>
        <li>Collect and prepare model forcings using the <code>exact extract</code> methodology.</li>
        <li>Create the simulation configuration files needed to run a model simulation.</li>
        <li>Execute model simulations using the NGEN executable.</li>
    </ol>
</div>


While the `ngiab_data_cli` commands listed in this notebook can be executed directly within the notebook cells, it is recommended to execute them in a terminal window instead. A convenient way to do this is by opening a Jupyter Terminal and docking it next to the notebook for easier reference and workflow.

![Jupyter Screenshot](./img/jupyter-screenshot.png)

----

## 1. Collect HydroFabric Data

Open the [USGS map view](https://maps.waterdata.usgs.gov/) to identify a location of interest. Note the stream gage number because this will be used to collect HydroFabric data for our NGEN simulation. [USGS National Water Information System Map](https://maps.waterdata.usgs.gov/mapper/index.html)


![USGS Gage Location](./img/usgs-on-north-river-alabama.png)

<div style="
    background: linear-gradient(to right, #607d8b, #00bcd4);
    color: white;
    padding: 10px;
    border-radius: 10px;
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    box-shadow: 0 4px 8px rgba(0,0,0,0.05);
    margin: 0;
">
<i class="fas fa-map-marker-alt" style="color: black; margin-right: 8px;"></i>
<strong>Define the Site Number of Interest</strong>
<p>Please provide the <strong>USGS site number</strong> you want to focus on. This parameter will be referenced throughout the notebook.</p>


In [ ]:
usgs_site_number = '02464000'

<br>
Subset the HydroFabric at our desired gage using the following command:

```
python -m ngiab_data_cli -i gage-02464000 --subset
```

#### Command Breakdown

<table style="border-collapse: collapse; width: 100%; font-size: 14px;">
  <thead>
    <tr style="background-color: navy; color: white; direction: ltr;">
      <th style="padding: 8px; text-align: left;">Argument</th>
      <th style="padding: 8px; text-align: left;">Description</th>
    </tr>
  </thead>
  <tbody style="direction: ltr;">
    <tr><td style="padding: 8px; text-align: left;"><code>-m ngiab_data_cli</code></td><td style="padding: 8px; text-align: left;">Tells Python which module it should run</td></tr>
    <tr><td style="padding: 8px; text-align: left;"><code>-i gage-02464000</code></td><td style="padding: 8px; text-align: left;">The input feature to subset</td></tr>
    <tr><td style="padding: 8px; text-align: left;"><code>--subset</code></td><td style="padding: 8px; text-align: left;">Indicates that the HydroFabric subsetting operation should be performed</td></tr>
  </tbody>
</table>

<div style="background-color:#444444; border-left: 4px solid #6c63ff; padding: 12px; border-radius: 6px; font-family: 'Fira Code', monospace; font-size: 14px; color: #ffffff;">
<p style="margin: 0;"><strong> Tip:</strong> You can also execute the same command within the notebook using Python syntax. Note that this only works if both the variable definition and the shell command are run within the same code cell. </p>
<pre style="margin-top: 10px;"><code>usgs_site_number = '02464000'

!python -m ngiab_data_cli -i gage-{usgs_site_number} --subset</code></pre>
</div>

We can preview these data using `Fiona` and `GeoPandas`.

In [ ]:
# define the path to our geopackage file
hf_filepath = f"./gage-{usgs_site_number}/config/gage-{usgs_site_number}_subset.gpkg"

Since our geopackage may contain more than one layer, we'll first use `fiona` to list the layers that are available.

In [ ]:
layers = fiona.listlayers(hf_filepath)

print( 45*'-' + '\nThe Geopackage contains the following layers: \n' + 45*'-')
for layer in layers:
    print(layer)

Let's load and plot the `divides`, `flowpaths`, and `nexus` layers.

In [ ]:
# load the layers
divides = gpd.read_file(hf_filepath, layer="divides")
flowpaths = gpd.read_file(hf_filepath, layer="flowpaths")
nexus = gpd.read_file(hf_filepath, layer="nexus")

print(f'{len(divides)} Divides')
print(f'{len(flowpaths)} Flowpaths')
print(f'{len(nexus)} Nexus')

You can either use `matplotlib` for a quick static visualization, or use `folium` for a more interactive experience that allows you to explore data layers and their associated attributes.

In [ ]:
# Create a figure using matplotlib
fig, ax = plt.subplots(figsize=(10, 10))

divides.plot(ax=ax, color='green', alpha=0.3, edgecolor='darkgreen')
flowpaths.plot(ax=ax, color='blue', edgecolor='blue', alpha=0.5, label='HydroFabric Reaches')
nexus.plot(ax=ax, color='red', edgecolor='red', alpha=0.5, label='HydroFabric Nexus')

plt.title(f'NGEN HydroFabric at USGS:{usgs_site_number}')
ax.legend()
plt.show()

To use `folium`, you need to make sure your spatial data is projected in **WGS84** (EPSG:4326), as folium requires geographic coordinates (latitude and longitude) for rendering layers correctly on the web map. Run the following code cell to generate the interactive map. Once the map appears:
- Zoom in to explore the area in more detail.
- Hover over the different catchments to highlight them.
- Click on a catchment, flow path, or nexus point to view their associated attributes. Note that Catchment IDs start with `cat-`, flow path IDs start with `wb-`, and Nexus point IDs start with `nex-`.

In [ ]:
# Read and reproject the data to EPSG:4326 (WGS84)
divides = gpd.read_file(hf_filepath, layer="divides").to_crs(epsg=4326)
flowpaths = gpd.read_file(hf_filepath, layer="flowpaths").to_crs(epsg=4326)
nexus = gpd.read_file(hf_filepath, layer="nexus").to_crs(epsg=4326)

# Compute center of map
bounds = divides.total_bounds
center = [(bounds[1] + bounds[3]) / 2, (bounds[0] + bounds[2]) / 2]

# Create base map
m = folium.Map(location=center, zoom_start=11, tiles="CartoDB positron")

def valid_fields(gdf):
    return [col for col in gdf.columns if col != 'geometry']

# Add divides with popup and highlight on click
folium.GeoJson(
    divides,
    name='Divides',
    style_function=lambda x: {
        'fillColor': 'green', 'color': 'darkgreen', 'weight': 1, 'fillOpacity': 0.3
    },
    highlight_function=lambda x: {
        'fillColor': '#ffffcc', 'color': 'orange', 'weight': 3, 'fillOpacity': 0.7
    },
    popup=folium.GeoJsonPopup(
        fields=valid_fields(divides),
        aliases=valid_fields(divides),
        labels=True,
        sticky=True,  # Ensures popup remains when clicked
        max_width=300
    )
).add_to(m)

# Add flowpaths with popup and highlight on click
folium.GeoJson(
    flowpaths,
    name='Flowpaths',
    style_function=lambda x: {
        'color': 'blue', 'weight': 2, 'opacity': 0.5
    },
    highlight_function=lambda x: {
        'color': 'red', 'weight': 3, 'opacity': 0.9
    },
    popup=folium.GeoJsonPopup(
        fields=valid_fields(flowpaths),
        aliases=valid_fields(flowpaths),
        labels=True,
        sticky=True,
        max_width=300
    )
).add_to(m)

# Add nexus points as red circles with popup
for _, row in nexus.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=5,
        color='red',
        fill=True,
        fill_color='red',
        fill_opacity=0.8,
        popup=folium.Popup(
            html="<br>".join([
                f"<b>{col}:</b> {row[col]}" for col in nexus.columns if col != "geometry"
            ]),
            max_width=300
        )
    ).add_to(m)

# Add layer control
folium.LayerControl().add_to(m)

# Show the map
m


Each of these GeoPandas dataframes contains attributes that we can look at as well.

In [ ]:
divides

We can attach additional attributes to these divide features.

In [ ]:
divide_attrs = gpd.read_file(hf_filepath, layer="divide-attributes")
merged_divides = divides.merge(divide_attrs, on='divide_id')

merged_divides

Notice there are 50 columns as opposed to the 11 in the original dataframe. We list all of the columns too.

In [ ]:
for c in merged_divides.columns:
    print(c)

We can do the same thing for the flowpaths.

In [ ]:
flow_attrs = gpd.read_file(hf_filepath, layer="flowpath-attributes")
merged_flows = flowpaths.merge(flow_attrs, on='id')
for c in merged_flows.columns:
    print(c)

___
## 2. Collect Model Forcings

Now that we have an NGEN HydroFabric for our location of interest, we can collect meteorlogical forcing data. This can be done using the `NGIAB Data Preprocessor`.

```
python -m ngiab_data_cli -i gage-02464000 -f --start 2022-01-01 --end 2022-12-31
```

#### Command Breakdown

<table style="border-collapse: collapse; width: 100%; font-size: 14px;">
  <thead>
    <tr style="background-color: navy; color: white; direction: ltr;">
      <th style="padding: 8px; text-align: left;">Argument</th>
      <th style="padding: 8px; text-align: left;">Description</th>
    </tr>
  </thead>
  <tbody style="direction: ltr;">
    <tr><td style="padding: 8px; text-align: left;"><code>-m ngiab_data_cli</code></td><td style="padding: 8px; text-align: left;">Specifies the Python module to run</td></tr>
    <tr><td style="padding: 8px; text-align: left;"><code>-i gage-02464000</code></td><td style="padding: 8px; text-align: left;">Defines the input feature ID (in this case, a USGS gage)</td></tr>
    <tr><td style="padding: 8px; text-align: left;"><code>-f</code></td><td style="padding: 8px; text-align: left;">Triggers the forcing data collection step</td></tr>
    <tr><td style="padding: 8px; text-align: left;"><code>--start 2022-01-01</code></td><td style="padding: 8px; text-align: left;">Sets the start date for data collection</td></tr>
    <tr><td style="padding: 8px; text-align: left;"><code>--end 2022-12-31</code></td><td style="padding: 8px; text-align: left;">Sets the end date for data collection</td></tr>
  </tbody>
</table>

<div style="background-color:#444444; border-left: 4px solid #6c63ff; padding: 12px; border-radius: 6px; font-family: 'Fira Code', monospace; font-size: 14px; color: #ffffff;">
<p style="margin: 0;"><strong> Tip:</strong> You can also execute the same command within the notebook using Python syntax:</p>
<pre style="margin-top: 10px;"><code>usgs_site_number = '02464000'

python -m ngiab_data_cli -i gage-02464000 -f --start 2022-01-01 --end 2022-12-31</code></pre>
</div>

We can preview these data using the `xarray` library.

In [ ]:
forcing_path = f'./gage-{usgs_site_number}/forcings/forcings.nc'

In [ ]:
ds = xarray.open_dataset(forcing_path)
ds

<table style="border-collapse: collapse; width: 100%; font-size: 14px;">
  <caption style="caption-side: top; font-weight: bold; font-size: 16px; padding: 8px; text-align: left;">
     Variable Reference Table
  </caption>
  <thead>
    <tr style="background-color: navy; color: white; direction: ltr;">
      <th style="padding: 8px; text-align: left;">Variable Name</th>
      <th style="padding: 8px; text-align: left;">Description</th>
      <th style="padding: 8px; text-align: left;">Units</th>
    </tr>
  </thead>
  <tbody style="direction: ltr;">
    <tr><td style="padding: 8px; text-align: left;"><code>APCP_surface</code></td><td style="padding: 8px; text-align: left">Precipitation at the surface level</td><td style="padding: 8px; text-align: left">Meters (m)</td></tr>
    <tr><td style="padding: 8px; text-align: left;"><code>DLWRF_surface</code></td><td style="padding: 8px; text-align: left">Downward Longwave Radiation Flux at the surface</td><td style="padding: 8px; text-align: left">W/m²</td></tr>
    <tr><td style="padding: 8px; text-align: left;"><code>PRES_surface</code></td><td style="padding: 8px; text-align: left">Air Pressure at the surface</td><td style="padding: 8px; text-align: left">Pascals (Pa)</td></tr>
    <tr><td style="padding: 8px; text-align: left;"><code>SPFH_2maboveground</code></td><td style="padding: 8px; text-align: left">Specific Humidity at 2 meters above ground</td><td style="padding: 8px; text-align: left">kg/kg</td></tr>
    <tr><td style="padding: 8px; text-align: left;"><code>precip_rate</code></td><td style="padding: 8px; text-align: left">Precipitation rate at the surface</td><td style="padding: 8px; text-align: left">m/s</td></tr>
    <tr><td style="padding: 8px; text-align: left;"><code>DSWRF_surface</code></td><td style="padding: 8px; text-align: left">Downward Shortwave Radiation Flux at the surface</td><td style="padding: 8px; text-align: left">W/m²</td></tr>
    <tr><td style="padding: 8px; text-align: left;"><code>TMP_2maboveground</code></td><td style="padding: 8px; text-align: left">Temperature at 2 meters above ground</td><td style="padding: 8px; text-align: left">Kelvin (K)</td></tr>
    <tr><td style="padding: 8px; text-align: left;"><code>UGRD_10maboveground</code></td><td style="padding: 8px; text-align: left">Eastward wind at 10 meters</td><td style="padding: 8px; text-align: left">m/s</td></tr>
    <tr><td style="padding: 8px; text-align: left;"><code>VGRD_10maboveground</code></td><td style="padding: 8px; text-align: left">Northward wind at 10 meters</td><td style="padding: 8px; text-align: left">m/s</td></tr>
  </tbody>
</table>


Check the dimensions and coordinates of the dataset

In [ ]:
ds.dims

In [ ]:
ds.coords

<div style="border-left: 4px solid #007acc; background-color: #eef6fb; padding: 10px; border-radius: 4px;">
    <p> <strong>Note:</strong>
       The dataset currently has no defined coordinates. While it includes variables like `Time` and `ids`, they are stored as regular data variables rather than coordinates. Having coordinates is important because they provide meaningful labels for dimensions such as time and spatial identifiers (e.g., catchment IDs). Coordinates allow for easier indexing, plotting, and alignment of data across variables. Let's assign appropriate coordinates to enhance the usability of the dataset.
    </p>
</div>


In [ ]:
# create a coordinate containing the values of `ids` and `Time`
ds = ds.assign_coords({'catchment-id': ds.ids})

# use the first row of Time (b/c is is not a 1D array) and convert into human-readable datetime values
import pandas as pd
time_vals = pd.to_datetime(ds['Time'].isel({'catchment-id': 0}).values, unit='s', origin='unix')

# Assign this as a 1D coordinate on the 'time' dimension
ds = ds.assign_coords(time=('time', time_vals))

We can visualize these data at any of the catchments within our domain. This is done by isolating our area of interest using xarray indexing, and then leveraging matplotlib to construct the plot. 

We'll need to provide a `catchment-id` for the data we want to plot. All of the catchment ids can be printed using the following command:

```
ds.ids.values
```

Alternatively, we can copy the catchment id associated with the outlet of our domain from the `ngiab_data_cli` output in the terminal, i.e. `cat-485431`

In [ ]:
outlet_catchment_id = 'cat-485431'

In [ ]:
ds_cat = ds.sel({"catchment-id": outlet_catchment_id})
ds_cat

In [ ]:
top_variable = 'precip_rate'
bottom_variable = 'TMP_2maboveground'

fig, ax1 = plt.subplots(figsize=(10, 5))

ds_cat[bottom_variable].plot(ax=ax1, label=bottom_variable, color='grey')
bottom_variable_units = ds_cat[bottom_variable].attrs['units']
ax1.set_ylabel(f'{bottom_variable} ({bottom_variable_units})')
ax1.tick_params(axis='y')
ax1.set_ylim((ds_cat[bottom_variable].min().item(),
              ds_cat[bottom_variable].max().item() * 1.1)) 

# Create a second y-axis
ax2 = ax1.twinx()

# Second series on right y-axis
ds_cat[top_variable].plot(ax=ax2, label=top_variable, color='blue')
top_variable_units = ds_cat[top_variable].attrs['units']
ax2.set_ylabel(f'{top_variable} ({top_variable_units})')
ax2.tick_params(axis='y')
ax2.set_ylim((ds_cat[top_variable].min(),
              ds_cat[top_variable].max().item() * 4)) 
ax2.invert_yaxis()

plt.tight_layout()
plt.show()

___
## 3. Generate Model Configuration Files

Now that we have an NGEN HydroFabric for our location of interest and the forcing data for our time period, we're ready to configure the model using the `NGIAB Data Preprocessor`.

```
python -m ngiab_data_cli -i gage-02464000 -r --start 2022-01-01 --end 2022-12-31
```


#### Command Breakdown

<table style="border-collapse: collapse; width: 100%; font-size: 14px;">
  <thead>
    <tr style="background-color: navy; color: white; direction: ltr;">
      <th style="padding: 8px; text-align: left;">Argument</th>
      <th style="padding: 8px; text-align: left;">Description</th>
    </tr>
  </thead>
  <tbody style="direction: ltr;">
    <tr><td style="padding: 8px; text-align: left;"><code>-m ngiab_data_cli</code></td><td style="padding: 8px; text-align: left;">Tells Python which module it should run</td></tr>
    <tr><td style="padding: 8px; text-align: left;"><code>-i gage-02464000</code></td><td style="padding: 8px; text-align: left;">Defines the input feature ID (in this case, a USGS gage)</td></tr>
    <tr><td style="padding: 8px; text-align: left;"><code>-r</code></td><td style="padding: 8px; text-align: left;">Indicates that the configuration files should be created</td></tr>
    <tr><td style="padding: 8px; text-align: left;"><code>--start 2022-01-01</code></td><td style="padding: 8px; text-align: left;">Sets the start date for data collection</td></tr>
    <tr><td style="padding: 8px; text-align: left;"><code>--end 2022-12-31</code></td><td style="padding: 8px; text-align: left;">Sets the end date for data collection</td></tr>
  </tbody>
</table>

<div style="background-color:#444444; border-left: 4px solid #6c63ff; padding: 12px; border-radius: 6px; font-family: 'Fira Code', monospace; font-size: 14px; color: #ffffff;">
<p style="margin: 0;"><strong> Tip:</strong> You can also execute the same command within the notebook using Python syntax:</p>
<pre style="margin-top: 10px;"><code>usgs_site_number = '02464000'

!python -m ngiab_data_cli -i gage-{usgs_site_number} -r --start 2022-01-01 --end 2022-12-31</code></pre>
</div>

List the configuration files that were generate by this previous command:

```
tree "$(pwd)/gage-02464000/config"
```

<div style="background-color:#444444; border-left: 4px solid #6c63ff; padding: 12px; border-radius: 6px; font-family: 'Fira Code', monospace; font-size: 14px; color: #ffffff;">
<p style="margin: 0;"><strong> Tip:</strong> You can also execute the same command within the notebook using Python syntax:</p>
<pre style="margin-top: 10px;"><code>usgs_site_number = '02464000'

!tree gage-{usgs_site_number}/config</code></pre>
</div>

### Explore Model Configuration Files

This directory contains **three** primary types of output files generated during the NGEN configuration process. Each plays a different role in defining how the model operates. These include (1) `cat_config/` (catchment-level model configuration for CFE and NOAH-OWP-M modules), (2) `realization.json` (model coupling and modular composition), and (3) `troute.yaml` (channel routing configuration for T-Route module). 

```text
├── cat_config
│   ├── CFE
│   │   ├── cat-485420.ini
│   │   └── ...
│   └── NOAH-OWP-M
│       ├── cat-485420.input
│       └── ...
├── gage-02464000_subset.gpkg
├── realization.json
└── troute.yaml
```


![Model Formulation](./img/model-formulation.png)

#### ● cat_config/CFE/*

The `CFE/` folder contains configuration files for the Conceptual Functional Equivalent (CFE) model. Each file (e.g., `cat-485420.ini`) defines **initialization** settings for an individual catchment. These include: state parameters (e.g., soil depth), adjustable parameters (e.g., maximum soil moisture for each soil layer), and temporal settings. These values are extracted directly from the geopackage produced in the first task of this notebook. 

Let's do a quick sanity check. As you might recall, we previously created a DataFrame called `merged_divides`, which includes additional attributes associated with each divide feature. The code below plots the values of these attributes across the 66 catchments.

In [ ]:
# Drop geometry
df_no_geom = merged_divides.drop(columns='geometry')

# Keep only numeric columns
df_numeric = df_no_geom.select_dtypes(include='number')

# Identify constant columns
constant_cols = df_numeric.columns[df_numeric.nunique() <= 1]
print("Constant columns:", list(constant_cols))

# Set titles just for numeric columns
titles = [f"{col} (CONSTANT)" if col in constant_cols else col for col in df_numeric.columns]

# Plot
axes = df_numeric.plot(
    subplots=True,
    layout=(-1, 2),         
    figsize=(16, 24),       
    title=titles,
    legend=False             
)

# Label settings
for ax in axes.flatten():
    ax.tick_params(axis='y', labelrotation=0, labelsize=14) 
    ax.tick_params(axis='x', labelsize=14) 
    ax.set_title(ax.get_title(), fontsize=18)

plt.tight_layout()
plt.show()


Note that the value for `mean.refkdt` is constant across all catchments (set to 2.0). This parameter, `refkdt`, plays a key role in the surface runoff formulation. It is a tunable coefficient that influences surface infiltration and, as a result, affects how total runoff is partitioned into surface and subsurface components. Let’s confirm where this value is coming from by checking the initialization files generated in this step.

```
grep "refkdt" "$(pwd)/gage-02464000/config/cat_config/CFE/cat-485420.ini"
```

#### ● cat_config/NOAH-OWP-M/*

The `NOAH-OWP-M/` folder contains input files for the refactored Noah-MP land surface model. Each `.input` file provides parameters specific to NOAA's version of the Noah-MP model. These input files provide land surface parameters related to vegetation, soil, snow, and energy balance modules.

#### ● realization.json

The file specifies how different model components are configured and coupled. Let's take a look at its contents.

In [ ]:
import json
from IPython.display import JSON, display

with open("./gage-02464000/config/realization.json") as f:
    data = json.load(f)

display(JSON(data))

At the root level of the JSON file, you will see three collapsible sections: `global`, `time`, and `routing`. Let's first explore the `time` and `routing` sections, as they contain less information.

<ul style="padding-left: 20px;">
  <li>
    Under the <code>time</code> section, you'll find parameters such as <code>start_time</code>, <code>end_time</code>, and <code>output_interval</code>, which is set to 3600 seconds (i.e., hourly outputs). The start and end dates here should match the values you have defined earlier.
  </li>
  <li>
    The <code>routing</code> section includes a reference to another configuration file, specified as a path to a YAML file.
    <br><em>(Quick note: while both YAML and JSON are used for structured data, YAML is often more human-readable due to its indentation-based syntax, whereas JSON is stricter and uses curly braces and commas.)</em>
  </li>
  <li>
    Under the <code>global</code> section, there are two sub-keys: <code>forcing</code> and <code>formulations</code>. 
    <ul style="padding-left: 20px; list-style-type: circle;">
      <li>
        The <code>forcing</code> key points to the meteorological forcing data path that we generated in previous steps.
      </li>
      <li>
        The <code>formulations</code> key provides details on how model components within the NextGen framework are coupled using the Basic Modeling Interface (BMI) via the <code>bmi_multi</code> mechanism.
      </li>
    </ul>
  </li>
</ul>


If you expand further into formulations, you will see a modules key with three items—indexed as 0, 1, and 2, each representing a different model component in the configuration.

In [ ]:
import json
from IPython.display import JSON, display

with open("./gage-02464000/config/realization.json") as f:
    data = json.load(f)
modules = data['global']['formulations'][0]['params']['modules']
display(JSON(modules))

<p>Three modules are connected in sequence:</p>

<ol start="0" style="padding-left: 20px;">
  <li>
    <a href="https://github.com/NOAA-OWP/SLoTH" target="_blank"><strong>SLoTH</strong></a> (Simple Logical Tautology Handler): 
    A simple model that assumes something is true because you say it is. For example, you can use SLoTH to provide a constant value (like <code>0</code> for <code>soil_ice_fraction</code>) instead of computing it via a complex model.
  </li>
  <li>
    <a href="https://github.com/NOAA-OWP/noah-owp-modular" target="_blank"><strong>Noah-OWP</strong></a> (refactored Noah-MP): 
    Simulates land–atmosphere interactions and generates surface runoff (<code>QINSUR</code>).
  </li>
  <li>
    <a href="https://github.com/NOAA-OWP/cfe/blob/master/MODEL.md" target="_blank"><strong>CFE</strong></a> (Conceptual Functional Equivalent): 
    A conceptual model that handles terrain routing and outputs lateral flow (<code>Q_OUT</code>) using the surface runoff from Noah-OWP.
  </li>
</ol>


<p><strong>Note:</strong> The order of modules in the realization file matters.</p>

<p>See another realization example: 
<a href="https://github.com/CIROH-UA/NGIAB_data_preprocess/blob/main/modules/data_sources/cfe-nowpm-realization-template.json" target="_blank">
CIROH-UA realization template
</a></p>

<p>More about realization configuration: 
<a href="https://noaa-owp.github.io/ngen/md_doc_2_r_e_a_l_i_z_a_t_i_o_n___c_o_n_f_i_g_u_r_a_t_i_o_n.html" target="_blank">
NextGen realization documentation
</a></p>


#### ● troute.yaml

This file contains settings for channel and reservoir routing using T-Route, a dynamic 1D river network model. 

<p>
  <a href="https://github.com/NOAA-OWP/t-route" target="_blank"><strong>T-Route</strong></a>
  computes streamflow by solving routing equations across a vector-based river network. 
  It takes runoff inputs from catchments and routes them through the stream or river network, 
  accounting for travel time and attenuation.
</p>

<p>
  T-Route supports multiple routing physics schemes, allowing users to select between simpler 
  hydrologic routing (e.g., Muskingum-Cunge) or more detailed hydraulic routing 
  (e.g., diffusive wave) to strike a balance between computational speed and accuracy. 
  This flexibility makes it possible to apply less intensive methods for headwater streams, 
  while using advanced techniques on main stems where backwater effects are more significant.
</p>

<p>
  For detailed documentation, refer to the 
  <a href="https://github.com/NOAA-OWP/t-route/blob/master/doc/v3_doc.yaml" target="_blank">
    T-Route configuration documentation
  </a>.
</p>


Let's load the yaml file and explore its keys. 

In [ ]:
import yaml
from IPython.display import display, JSON

with open("./gage-02464000/config/troute.yaml", 'r') as f:
    data = yaml.safe_load(f)

params = data

display(JSON(params))


Under the `compute_parameters` section, the value of `compute_kernel` is set to "V02-structured". While this label may seem vague at first, referring to the T-Route documentation shows that this setting indicates the use of the Muskingum-Cunge method as the routing engine for the simulation.

Under the `network_topology_parameters` section, information about the flowlines is sourced from the flowpaths layer of the GeoPackage. The following code displays the range of values for each parameter across the flowlines. To learn more about the meaning and role of each parameter, refer to the T-Route documentation.

As a starting point, can you identify what **musK** and **musX** represent?

In [ ]:
# Drop geometry
df_no_geom = merged_flows.drop(columns='geometry')

# Keep only numeric columns
df_numeric = df_no_geom.select_dtypes(include='number')

# Identify constant columns
constant_cols = df_numeric.columns[df_numeric.nunique() <= 1]
print("Constant columns:", list(constant_cols))

# Set titles just for numeric columns
titles = [f"{col} (CONSTANT)" if col in constant_cols else col for col in df_numeric.columns]

# Plot
axes = df_numeric.plot(
    subplots=True,
    layout=(-1, 2),          
    figsize=(16, 24),        
    title=titles,
    legend=False            
)

# Label settings
for ax in axes.flatten():
    ax.tick_params(axis='y', labelrotation=0, labelsize=14) 
    ax.tick_params(axis='x', labelsize=14) 
    ax.set_title(ax.get_title(), fontsize=18)

plt.tight_layout()
plt.show()


Print the Manning's roughness of main channel for flowpath ID wb-485420.

In [ ]:
merged_flows[merged_flows['id']=='wb-485420'].n

___
## 4. Executing a Simulation

Move into the directory where our model simulation data exists.

```
cd ./gage-02464000
```

Run the simulation using the `ngen` command. This command requires several inputs:

`ngen <catchment_data_path> <catchment subset ids> <nexus_data_path> <nexus subset ids> <realization_config_path>`


#### Command Breakdown

<table style="border-collapse: collapse; width: 100%; font-size: 14px;">
  <thead>
    <tr style="background-color: navy; color: white; direction: ltr;">
      <th style="padding: 8px; text-align: left;">Argument</th>
      <th style="padding: 8px; text-align: left;">Description</th>
    </tr>
  </thead>
  <tbody style="direction: ltr;">
    <tr><td style="padding: 8px; text-align: left;"><code>--catchment_data_path</code></td><td style="padding: 8px; text-align: left;">Path to the catchment input data</td></tr>
    <tr><td style="padding: 8px; text-align: left;"><code>--catchment_subset_ids</code></td><td style="padding: 8px; text-align: left;">Comma-separated list of catchment IDs to include, or <code>""</code> to include all</td></tr>
    <tr><td style="padding: 8px; text-align: left;"><code>--nexus_data_path</code></td><td style="padding: 8px; text-align: left;">Path to the nexus input data</td></tr>
    <tr><td style="padding: 8px; text-align: left;"><code>--nexus_subset_ids</code></td><td style="padding: 8px; text-align: left;">Comma-separated list of nexus IDs to include, or <code>""</code> to include all</td></tr>
    <tr><td style="padding: 8px; text-align: left;"><code>--realization_config_path</code></td><td style="padding: 8px; text-align: left;">Path to the realization configuration file for the simulation</td></tr>
  </tbody>
</table>

</div>

For example

```
ngen ./config/gage-02464000_subset.gpkg "" \
     ./config/gage-02464000_subset.gpkg "" \
     ./config/realization.json
```

<div style="background-color:#444444; border-left: 4px solid #6c63ff; padding: 12px; border-radius: 6px; font-family: 'Fira Code', monospace; font-size: 14px; color: #ffffff;">
<p style="margin: 0;"><strong> Tip:</strong> You can also execute the same command within the notebook using Python syntax:</p>
<pre style="margin-top: 10px;"><code>usgs_site_number = '02464000'

!ngen ./config/gage-{usgs_site_number}_subset.gpkg "" ./config/gage-{usgs_site_number}_subset.gpkg "" ./config/realization.json
</div>

___
## 5. Evaluating Simulation Results

Our NextGen simulation has created a number of different outputs. These outputs are categorized as either (1) NextGen model outputs or (2) T-route model outputs; all of them can be found in the `outputs` directory. Below is a high-level summary of the output variables that our simulation produced.


##### NextGen Model Outputs

<table style="border-collapse: collapse; width: 100%; font-size: 14px;">
  <thead>
    <tr style="background-color: navy; color: white; direction: ltr;">
      <th style="padding: 8px; text-align: left;">Variable Name</th>
      <th style="padding: 8px; text-align: left;">Description</th>
      <th style="padding: 8px; text-align: left;">Units</th>
    </tr>
  </thead>
  <tbody style="direction: ltr;">
    <tr>
      <td style="padding: 8px; text-align: left;">RAIN_RATE</td>
      <td style="padding: 8px; text-align: left;">Rainfall Rate</td>
      <td style="padding: 8px; text-align: left;">Meters (m)</td>
    </tr>
    <tr>
      <td style="padding: 8px; text-align: left;">GIUH_RUNOFF</td>
      <td style="padding: 8px; text-align: left;">Lagged and attenuated runoff using the Geomorphological Instantaneous Unit Hydrograph</td>
      <td style="padding: 8px; text-align: left;">Meters (m)</td>
    </tr>
    <tr>
      <td style="padding: 8px; text-align: left;">INFILTRATION_EXCESS</td>
      <td style="padding: 8px; text-align: left;">Amount of rainfall that exceeds infiltration capacity</td>
      <td style="padding: 8px; text-align: left;">Meters (m)</td>
    </tr>
    <tr>
      <td style="padding: 8px; text-align: left;">DIRECT_RUNOFF</td>
      <td style="padding: 8px; text-align: left;">Surface runoff from rainfall/throughfall input</td>
      <td style="padding: 8px; text-align: left;">Meters (m)</td>
    </tr>
    <tr>
      <td style="padding: 8px; text-align: left;">NASH_LATERAL_RUNOFF</td>
      <td style="padding: 8px; text-align: left;">Lateral subsurface flow through the Nash cascade</td>
      <td style="padding: 8px; text-align: left;">Meters (m)</td>
    </tr>
    <tr>
      <td style="padding: 8px; text-align: left;">DEEP_GW_TO_CHANNEL_FLUX</td>
      <td style="padding: 8px; text-align: left;">Flux from deep groundwater to the channel</td>
      <td style="padding: 8px; text-align: left;">Meters (m)</td>
    </tr>
    <tr>
      <td style="padding: 8px; text-align: left;">SOIL_TO_GW_FLUX</td>
      <td style="padding: 8px; text-align: left;">Flux from soil to groundwater</td>
      <td style="padding: 8px; text-align: left;">Meters (m)</td>
    </tr>
    <tr>
      <td style="padding: 8px; text-align: left;">Q_OUT</td>
      <td style="padding: 8px; text-align: left;">Outflow from the catchment</td>
      <td style="padding: 8px; text-align: left;">Meters (m)</td>
    </tr>
    <tr>
      <td style="padding: 8px; text-align: left;">POTENTIAL_ET</td>
      <td style="padding: 8px; text-align: left;">Potential evapotranspiration</td>
      <td style="padding: 8px; text-align: left;">Meters (m)</td>
    </tr>
    <tr>
      <td style="padding: 8px; text-align: left;">ACTUAL_ET</td>
      <td style="padding: 8px; text-align: left;">Actual evapotranspiration</td>
      <td style="padding: 8px; text-align: left;">Meters (m)</td>
    </tr>
    <tr>
      <td style="padding: 8px; text-align: left;">GW_STORAGE</td>
      <td style="padding: 8px; text-align: left;">Groundwater storage</td>
      <td style="padding: 8px; text-align: left;">Meters (m)</td>
    </tr>
    <tr>
      <td style="padding: 8px; text-align: left;">SOIL_STORAGE</td>
      <td style="padding: 8px; text-align: left;">Soil moisture storage</td>
      <td style="padding: 8px; text-align: left;">Meters (m)</td>
    </tr>
    <tr>
      <td style="padding: 8px; text-align: left;">SOIL_STORAGE_CHANGE</td>
      <td style="padding: 8px; text-align: left;">Change in soil moisture storage</td>
      <td style="padding: 8px; text-align: left;">Meters (m)</td>
    </tr>
    <tr>
      <td style="padding: 8px; text-align: left;">SURF_RUNOFF_SCHEME</td>
      <td style="padding: 8px; text-align: left;">Surface runoff as determined by the runoff scheme</td>
      <td style="padding: 8px; text-align: left;">Unitless</td>
    </tr>
    <tr>
      <td style="padding: 8px; text-align: left;">NWM_PONDED_DEPTH</td>
      <td style="padding: 8px; text-align: left;">Ponded depth of water based on NWM model</td>
      <td style="padding: 8px; text-align: left;">Meters (m)</td>
    </tr>
  </tbody>
</table>

<p style="font-style: italic; font-size: 13px; margin-top: 8px;">Note: this metadata was collected by David Tarboton in 2025.</p>


##### T-Route Model Outputs

<table style="border-collapse: collapse; width: 100%; font-size: 14px;">
  <thead>
    <tr style="background-color: navy; color: white; direction: ltr;">
      <th style="padding: 8px; text-align: left;">Variable Name</th>
      <th style="padding: 8px; text-align: left;">Description</th>
      <th style="padding: 8px; text-align: left;">Units</th>
    </tr>
  </thead>
  <tbody style="direction: ltr;">
    <tr>
      <td style="padding: 8px; text-align: left;">feature_id</td>
      <td style="padding: 8px; text-align: left;">Reach segment identifier</td>
      <td style="padding: 8px; text-align: left;">Unitless</td>
    </tr>
    <tr>
      <td style="padding: 8px; text-align: left;">flow</td>
      <td style="padding: 8px; text-align: left;">River streamflow</td>
      <td style="padding: 8px; text-align: left;">Cubic Meters per Second (m³/s)</td>
    </tr>
    <tr>
      <td style="padding: 8px; text-align: left;">velocity</td>
      <td style="padding: 8px; text-align: left;">River velocity</td>
      <td style="padding: 8px; text-align: left;">Meters per Second (m/s)</td>
    </tr>
    <tr>
      <td style="padding: 8px; text-align: left;">depth</td>
      <td style="padding: 8px; text-align: left;">River depth</td>
      <td style="padding: 8px; text-align: left;">Meters (m)</td>
    </tr>
    <tr>
      <td style="padding: 8px; text-align: left;">nudge</td>
      <td style="padding: 8px; text-align: left;">River streamflow nudge values</td>
      <td style="padding: 8px; text-align: left;">Cubic Meters per Second (m³/s)</td>
    </tr>
  </tbody>
</table>


#### ● Exploring Simulated Streamflow Trends

Load the t-route output file into memory using `xarray`

In [ ]:
t_route_output = f'./gage-{usgs_site_number}/outputs/troute/troute_output_202201010000.nc'

In [ ]:
ds = xarray.open_dataset(t_route_output)
ds

We'll need to use a `feature_id` to query and plot simulation results from the t-route output file. One way that we can accomplish this through the relationships defined in the HydroFabric. Specifically, the `divides` and `flowpaths` both contain identifiers that can be used to derive the `feature_id` value.

In [ ]:
flowpaths

The `catchment_id` can be used to query the t-route outputs in the Xarray Dataset that we loaded above. The only difference between these fields is that the former contains a prefix of `cat-` which when removed maps directly to the `feature_id` variable in the t-route output. For example, a catchment value of `cat-485431` maps to a feature_id of `485431`.

In [ ]:
outlet_catchment_id = 'cat-485431'
outlet_feature_id = int(outlet_catchment_id.split('-')[-1])

print(f'The feature_id of the domain outlet = {outlet_feature_id}')

Now we can isolate the variables associated with this reach and create a plot.

In [ ]:
dat = ds.sel(feature_id = outlet_feature_id)

dat.flow.plot();

#### ● Comparison of Modeled and Observed Streamflow

Collect USGS streamflow observations and compare them with the modeled streamflow.

In [ ]:
usgs_site_number

In [ ]:
df_obs = nwis.get_record(sites=usgs_site_number, service='iv', start='2022-01-01', end='2022-12-31')
df_obs

Print the maximum streamflow value for the period of interest

In [ ]:
df_obs.max()

Plot the modeled streamflow against the observed value

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

# plot modeled streamflow
modeled_series = dat.flow
modeled_series.plot(ax=ax, color='grey', label='Modeled Streamflow')

# convert observed streamflow to cms and plot
observed_series = df_obs['00060'] / 3.28**3
observed_series.plot(ax=ax, color='green', label='Observed NWIS Streamflow')

ax.set_title(f"Modeled ({outlet_feature_id}) vs Observed ({usgs_site_number}) Streamflow")

plt.legend()
plt.grid()
plt.tight_layout()

#### ● Compute Streamflow Bias Duration Curve (BDC)

<div style="font-family: Arial, sans-serif; line-height: 1.6;">
  <p>
    As you may know, a <strong>Flow Duration Curve (FDC)</strong> represents the relationship between streamflow magnitude and its exceedance probability.
    Building on this concept, a <strong>Bias Duration Curve (BDC)</strong> typically refers to a plot that shows the difference between modeled and observed 
    streamflow values across the FDC exceedance spectrum. This helps visualize <em>systematic bias</em> in model performance across different flow regimes, such as low, medium, and high flows.
  </p>
  <p>
    A BDC is constructed by first calculating <strong>relative streamflow values</strong> (both simulated and observed), sorting them in descending order, 
    and then plotting the <strong>relative bias</strong> at each exceedance probability. The underlying theory suggests that bias depends on the 
    <em>probability distribution</em> of the modeled versus observed streamflow values.
  </p>
  <p>
    <strong>Exceedance probability</strong> in the context of a Flow Duration Curve (FDC) represents the likelihood that a given value (e.g., streamflow) 
    will be equaled or exceeded over a specified period. It is typically expressed as a percentage. Lower exceedance probability values correspond to higher streamflows that are rarely exceeded. These points appear on the 
    <strong>left-hand side</strong> of the FDC and are often associated with extreme events such as floods. 
    Conversely, higher exceedance probability values indicate lower streamflows that occur more frequently, 
    typically found on the <strong>right-hand side</strong> of the curve, reflecting baseflow or dry-period conditions.
  </p>
  <p>
    A <strong>Bias Duration Curve (BDC)</strong> provides insight into how model bias is distributed across the exceedance probability spectrum. 
    This helps us identify whether the model tends to overpredict or underpredict flow in different parts of the flow regime. 
    For example, it can help answer questions like: <em>Does the model overestimate during rare high-flow events?</em> 
    or <em>Is there consistent underprediction during low-flow periods?</em>
  </p>


In [ ]:
# resample observed to hourly timesteps and remove timezone from datetime index
observed_series_hourly_avg = observed_series.resample('h').mean()
observed_series_hourly_avg = observed_series_hourly_avg.tz_localize(None)
observed_series_hourly_avg

In [ ]:
print(f'Length of observed = {len(observed_series_hourly_avg)}')
print(f'Length of modeled = {len(modeled_series)}')

Ensure alignment and drop missing data

In [ ]:
# figure out where these two series have common dates
common_index = observed_series_hourly_avg.index.intersection(dat.to_dataframe().flow.index)

observed_series = observed_series_hourly_avg.loc[common_index]
modeled_series = dat.to_dataframe().flow.loc[common_index]

print(f'Length of observed = {len(observed_series)}')
print(f'Length of modeled = {len(modeled_series)}')

Calcualate exceedance probability

In [ ]:
# observed_series = observed_series.dropna()
sorted_modeled_flows = np.sort(modeled_series)[::-1]
sorted_obs_flows = np.sort(observed_series)[::-1]

# calculate exceedance probabilities
n = len(sorted_obs_flows)
rank = np.arange(1, n + 1)
exceedance = rank / (n + 1) * 100

In [ ]:
sorted_modeled_flows

Plot Bias Duration Curve

In [ ]:
# Compute relative bias
bias = (sorted_modeled_flows - sorted_obs_flows) / sorted_obs_flows  # Relative bias

# Plot Bias Duration Curve
plt.figure(figsize=(8, 5))
plt.plot(exceedance, bias * 100, label='Relative Bias (%)')
plt.axhline(0, color='black', linestyle='--')
plt.xscale('log')
plt.xlabel('Exceedance Probability (%)')
plt.ylabel('Relative Bias (%)')
plt.title('Bias Duration Curve (BDC)')
plt.grid(True, which='both', linestyle='--')
plt.legend()
plt.show()


<div style="border-left: 4px solid #007acc; background-color: #eef6fb; padding: 10px; border-radius: 4px;">
    <p> <strong>Key Insight:</strong>
        Bias pattern over exceedance probabilities helps identify systematic errors:
    </p>
    <ol>
        <li>Bias near 0% → Good agreement</li>
        <li>Positive bias → Model overpredicts</li>
        <li>Negative bias → Model underpredicts</li>
    </ol>
</div>
